# Prise en main de Cantera

Exécuter les cellules dans l'ordre avec **Maj + Entrée**, ou choisir **Run → Run All Cells**. Aucune installation n'est nécessaire dans cette session Binder.

Cet exemple calcule l'équilibre chimique adiabatique d'un mélange méthane/air à pression constante. Il utilise uniquement le mécanisme `gri30.yaml` fourni avec Cantera.

In [ ]:
import sys
from pathlib import Path

import cantera as ct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f'Python : {sys.version.split()[0]}')
print(f'Cantera : {ct.__version__}')

## Mélange initial et hypothèses

Le combustible est CH₄. L'air est représenté par un mélange O₂/N₂ de rapport molaire N₂/O₂ = 3,76, sans argon ni humidité. Le gaz est supposé idéal ; seules les espèces gazeuses de GRI-Mech 3.0 sont prises en compte.

La température initiale est **300 K** et la pression **1 atm = 101325 Pa**. La richesse φ est le rapport combustible/comburant divisé par sa valeur stœchiométrique : φ < 1 pour un mélange pauvre, φ = 1 à la stœchiométrie et φ > 1 pour un mélange riche.

`equilibrate('HP')` recherche l'équilibre en conservant l'enthalpie et la pression du mélange initial. Cela représente un état final adiabatique à pression constante. Ce modèle ne calcule ni la cinétique d'approche de l'équilibre, ni une structure ou une vitesse de flamme.

In [ ]:
T_initiale = 300.0  # K
pression = ct.one_atm  # Pa
combustible = 'CH4:1'
air = 'O2:1, N2:3.76'
phi = 1.0

gaz = ct.Solution('gri30.yaml')
gaz.TP = T_initiale, pression
gaz.set_equivalence_ratio(phi, fuel=combustible, oxidizer=air)
enthalpie_initiale = gaz.enthalpy_mass
gaz.equilibrate('HP')

print(f'Température adiabatique d’équilibre à φ = {phi:.2f} : {gaz.T:.1f} K')
print(f'Pression finale : {gaz.P:.0f} Pa')
print(f'Écart d’enthalpie massique : {gaz.enthalpy_mass - enthalpie_initiale:.3e} J/kg')

especes = ['CH4', 'O2', 'N2', 'CO2', 'H2O', 'CO', 'H2', 'NO']
pd.DataFrame({
    'Espèce': especes,
    'Fraction molaire à l’équilibre': [gaz[nom].X[0] for nom in especes],
})

## Influence de la richesse

On fait varier φ de 0,5 à 1,5. **Le mélange et la température initiale sont réinitialisés avant chaque calcul**, afin de comparer des états initiaux à la même température et à la même pression. Les températures obtenues sont des résultats du modèle d'équilibre, sans pertes thermiques.

In [ ]:
richesses = np.linspace(0.5, 1.5, 31)
temperatures = []

for richesse in richesses:
    gaz.TP = T_initiale, pression
    gaz.set_equivalence_ratio(richesse, fuel=combustible, oxidizer=air)
    gaz.equilibrate('HP')
    temperatures.append(gaz.T)

resultats = pd.DataFrame({
    'richesse_phi': richesses,
    'temperature_equilibre_K': temperatures,
    'temperature_initiale_K': T_initiale,
    'pression_Pa': pression,
})

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(richesses, temperatures, 'o-', markersize=3)
ax.set_xlabel('Richesse φ (sans dimension)')
ax.set_ylabel('Température adiabatique d’équilibre (K)')
ax.set_title('CH₄ / air — état initial : 300 K, 1 atm')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Exporter et conserver les résultats

La cellule suivante crée un fichier CSV et une figure PNG. Pour les télécharger, faire un clic droit sur chaque fichier dans l'explorateur à gauche, puis **Download**. Enregistrer et télécharger aussi ce notebook si vous l'avez modifié. **Les fichiers présents sur Binder seront perdus à la fin de la session.**

In [ ]:
resultats.to_csv('resultats_equilibre.csv', index=False)
fig.savefig('temperature_equilibre.png', dpi=160)
print('Fichiers à télécharger :')
for nom in ['resultats_equilibre.csv', 'temperature_equilibre.png']:
    print(f'  {Path(nom).name}')